In [1]:
import pypandoc
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
import re



In [2]:
# Path to your Word document
doc_path = '/data/results/TrialRun/SummarizedArticles100ish.docx'

# Convert the document to Markdown
markdown_text = pypandoc.convert_file(doc_path, 'markdown')

In [3]:
markdown_text[:5000]

"'Spontaneous'\n=============\n\nSpontaneous intracranial hypotension (SIH) is a condition characterized\nby orthostatic headaches and is often associated with cerebrospinal\nfluid (CSF) leaks (Kanao-Kanda et al., 2022; Hughes & Chavez, 2022;\nFerraro et al., 2020; Lee et al., 2020; Peterson et al., 2021). The\ncondition is often underdiagnosed and can be challenging to manage due\nto its varied presentation and the potential absence of key signs\n(Giagkou et al., 2022; D'Amico et al., 2020). SIH can be associated with\nother conditions such as postural tachycardia syndrome (POTS) (Kato et\nal., 2019) and endolymphatic hydrops (Sakano et al., 2020).\n\nImaging techniques such as magnetic resonance imaging (MRI) and computed\ntomography (CT) are crucial in diagnosing SIH (Lee et al., 2019; Lin et\nal., 2020; Goddu Govindappa et al., 2017; Limaye et al., 2016). However,\nthese techniques may not always identify the CSF leak, making diagnosis\nchallenging (Lee et al., 2022; Hughes & Chave

In [4]:
def extract_apa_citations(markdown_text):
    # Split the document into paragraphs
    paragraphs = markdown_text.split("\n\n")

    # Filter paragraphs that contain "PMID"
    citations = [para for para in paragraphs if "PMID" in para]
    non_citations = [para for para in paragraphs if "PMID" not in para]

    return citations, non_citations

In [5]:
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [6]:
# Example usage
citations, non_citations = extract_apa_citations(markdown_text)

In [10]:

CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT16k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-35-turbo-16",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


SYSTEM_DRAFT_TEMPLATE = """
You are an expert medical writer. Your job is to write a draft of a scoping review article to address this question: {question}\n\n 

You will be given article from summaries (with APA-style citations) made from large sets of categorized articles.

Both the categories and summaries were made by experts.
"""

HUMAN_INTRODUCTION_TEMPLATE = """
Let's work on the introduction. Here are the category summaries with in-text citations:

---
{summaries}
---

Please provide a draft of the introduction for the scoping review article, using in-text citations where appropriate.

Format your response as markdown like this

# Introduction

## Background
List the question to be addressed

## Objectives
Clearly define the objectives of the scoping review.

## Significance
Significance of the findings of the review
"""


HUMAN_CONCLUSION_TEMPLATE = """
Let's work on the Conclusion. Here are the category summaries with in-text citations:

---
{summaries}
---

And here is the Introduction 

---
{introduction}
---

Please provide a draft of the conclusion for the scoping review article, using in-text citations where appropriate.

Format your response as markdown like this

# Conclusion
Summarize the key findings and their implications in a concise manner.

"""

HUMAN_ABSTRACT_TEMPLATE = """
Let's work on the Abstract. Here are the category summaries with in-text citations:

---
{summaries}
---

Here is the Introduction 

---
{introduction}
---

Here is the methdology

---
{methodology}
---
And here is the conclusion

---
{conclusion}
---

Please provide a draft of the abstract for the scoping review article.

Format your response as markdown like this

# Abstract
A brief summary of the review, including the purpose, methodology, main findings, and conclusions.
"""


draft_system_message_prompt = SystemMessagePromptTemplate.from_template(SYSTEM_DRAFT_TEMPLATE)

human_introduction_prompt = HumanMessagePromptTemplate.from_template(HUMAN_INTRODUCTION_TEMPLATE)
human_conclusion_prompt = HumanMessagePromptTemplate.from_template(HUMAN_CONCLUSION_TEMPLATE)
human_abstract_prompt = HumanMessagePromptTemplate.from_template(HUMAN_ABSTRACT_TEMPLATE)

draft_introduction_prompt = ChatPromptTemplate.from_messages([draft_system_message_prompt, human_introduction_prompt])
draft_conclusion_prompt = ChatPromptTemplate.from_messages([draft_system_message_prompt, human_conclusion_prompt])
draft_abstract_prompt = ChatPromptTemplate.from_messages([draft_system_message_prompt, human_abstract_prompt])



In [11]:
methodology_boilerplate = """
# Methodology
Paraphrase this methodology section: 

## Use of Generative AI in the Scoping Review Process
This scoping review leveraged a generative AI tool designed to streamline the process of identifying and synthesizing relevant literature. The tool's capabilities include generating PubMed queries based on user-provided research questions, returning a manageable list of articles for review, and assisting in the iterative refinement of the search process. It is noteworthy that this AI tool functions interactively, requiring user input at critical stages, particularly in the decision-making process for article inclusion or exclusion. Users, typically authors of the review, play a pivotal role in guiding the AI through iterative cycles of query refinement and article selection, ensuring that the final set of articles closely aligns with the specific research question. The availability and specific functionalities of this tool, if published, can be found in [insert source here].

## Information Sources
The primary information source for this scoping review was PubMed. The AI tool was programmed to search within specific parameters to optimize relevance and manageability. Limitations applied to the search included language restrictions (e.g., articles in English) and the availability of abstracts, which were necessary for the initial screening process.

## Selection Process
The selection process was a collaborative effort between the generative AI tool and the review authors. Initially, the AI generated a PubMed query based on the entered research question. Authors then reviewed the abstracts of the returned articles, marking those relevant to the scoping review. This feedback was used by the AI to refine the search query, incorporating insights from titles and keywords of the selected articles. This iterative process, involving repeated cycles of AI-generated searches and author screenings, continued until the authors were satisfied with the comprehensiveness and relevance of the articles identified.

## Synthesis of Results
The synthesis of results involved a piecemeal approach, allowing for the thorough examination of a large number of articles. After achieving a satisfactory collection of articles, the authors provided categorization criteria, and the AI assigned categories to each article. The authors reviewed and, if necessary, corrected these categorizations. Subsequently, the AI summarized the articles within each category, utilizing full texts when available. Authors played a critical role in this phase as well, reviewing and confirming or editing these summaries to ensure accuracy and coherence. This structured approach facilitated a comprehensive and nuanced synthesis of the results, tailored to the specific objectives of the scoping review.

This methodology underscores the symbiotic relationship between the generative AI tool and the human authors, combining the efficiency and analytical capabilities of AI with the critical and contextual expertise of researchers. The outcome is a meticulously curated and synthesized body of literature that addresses the defined research question in a comprehensive and targeted manner.

"""

In [12]:
introduction_result = CHAT.invoke(draft_introduction_prompt.format_prompt(
    question=user_question,
    summaries="\n\n".join(non_citations)
    ).to_messages())
print(introduction_result.content)

# Introduction

## Background
Post-dural spinal puncture headache (PDPH) is a common complication following spinal puncture procedures. It is characterized by an orthostatic headache that can be accompanied by neurological symptoms (Kamm & Forderreuther, 2021). PDPH is more common in young patients and is more frequently observed in women than men (Dahl & Rosenberg, 1990). The primary treatment for PDPH is the establishment of an epidural blood patch (EBP) (Dahl & Rosenberg, 1990). 

Spontaneous intracranial hypotension (SIH) is a condition characterized by orthostatic headaches and is often associated with cerebrospinal fluid (CSF) leaks (Kanao-Kanda et al., 2022; Hughes & Chavez, 2022; Ferraro et al., 2020). SIH can be associated with other conditions such as postural tachycardia syndrome (POTS) and endolymphatic hydrops (Kato et al., 2019; Sakano et al., 2020). Imaging techniques such as magnetic resonance imaging (MRI) and computed tomography (CT) are crucial in diagnosing SIH, but

In [14]:
conclusion_result = CHAT.invoke(draft_conclusion_prompt.format_prompt(
    question=user_question,
    summaries="\n\n".join(non_citations),
    introduction=introduction_result.content
    ).to_messages())
print(conclusion_result.content)

# Conclusion

This scoping review aimed to explore the association between a diagnosis of a connective tissue disease and the occurrence of post-dural spinal puncture headaches (PDPH). The review of the literature revealed several key findings.

First, spontaneous intracranial hypotension (SIH), a condition often associated with connective tissue diseases, was found to be a potential contributor to PDPH. SIH is characterized by orthostatic headaches and can be challenging to diagnose due to the absence of key signs and the limitations of imaging techniques. However, some studies suggest that underlying connective tissue disorders may contribute to the development of CSF leaks, which can lead to PDPH.

Second, connective tissue diseases such as systemic lupus erythematosus (SLE), Marfan syndrome, and hypermobility spectrum disorders were identified as potential risk factors for PDPH. Case studies highlighted the occurrence of significant headaches, including PDPH, in patients with these

In [15]:
abstract_result = CHAT.invoke(draft_abstract_prompt.format_prompt(
    question=user_question,
    summaries=markdown_text,
    introduction=introduction_result.content,
    methodology=methodology_boilerplate,
    conclusion=conclusion_result.content
    ).to_messages())
print(abstract_result.content)

# Abstract

This scoping review aimed to explore the association between a diagnosis of a connective tissue disease and the occurrence of post-dural spinal puncture headaches (PDPH). The review of the literature revealed several key findings. 

First, spontaneous intracranial hypotension (SIH), a condition often associated with connective tissue diseases, was found to be a potential contributor to PDPH. Second, connective tissue diseases such as systemic lupus erythematosus (SLE), Marfan syndrome, and hypermobility spectrum disorders were identified as potential risk factors for PDPH. Third, the literature emphasized the complexity of diagnosing conditions that present with headache symptoms, such as intracranial hypotension and meningitis, and the potential for misdiagnosis. 

Overall, the findings suggest that a diagnosis of a connective tissue disease may contribute to the occurrence of PDPH. However, further research is needed to confirm and better understand this association. The 

In [22]:
assembled_draft = (
    abstract_result.content + "\n\n" +
    introduction_result.content + "\n\n" + 
    methodology_boilerplate + "\n\n" + 
    "Results/Discussion" + "\n\n" +
    "\n\n".join(non_citations) + "\n\n" +
    "References" + "\n\n".join(citations)
    
)

In [23]:
print(assembled_draft)

# Abstract

This scoping review aimed to explore the association between a diagnosis of a connective tissue disease and the occurrence of post-dural spinal puncture headaches (PDPH). The review of the literature revealed several key findings. 

First, spontaneous intracranial hypotension (SIH), a condition often associated with connective tissue diseases, was found to be a potential contributor to PDPH. Second, connective tissue diseases such as systemic lupus erythematosus (SLE), Marfan syndrome, and hypermobility spectrum disorders were identified as potential risk factors for PDPH. Third, the literature emphasized the complexity of diagnosing conditions that present with headache symptoms, such as intracranial hypotension and meningitis, and the potential for misdiagnosis. 

Overall, the findings suggest that a diagnosis of a connective tissue disease may contribute to the occurrence of PDPH. However, further research is needed to confirm and better understand this association. The 